In [ ]:
"""
Playground: tokenization + generation/decoding with Qwen2.5-0.5B-Instruct.

Tip: run the numbered sections one at a time in a REPL / notebook so you can
poke at the variables. Everything below depends only on the SETUP block.

    pip install "transformers>=4.45" torch
"""

In [6]:
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

_mps_ok = torch.backends.mps.is_available() and os.environ.get("USE_MPS") == "1"
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if _mps_ok
    else "cpu"
)

# ----------------------------------------------------------------------
# SETUP: load once, reuse everywhere
# ----------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()  # disables dropout etc. — important for inference

print(f"Device:          {DEVICE}")
print(f"Vocab size:      {tokenizer.vocab_size}")
print(f"EOS token:       {tokenizer.eos_token!r} (id {tokenizer.eos_token_id})")
print(f"Special tokens:  {tokenizer.special_tokens_map}")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 7401.41it/s]


Device:          cpu
Vocab size:      151643
EOS token:       '<|im_end|>' (id 151645)
Special tokens:  {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}


In [3]:
import torch 
isinstance(model, torch.nn.Module)

True

In [9]:
# ======================================================================
# 1. TOKENIZATION BASICS
# ======================================================================
text = "Hello, world! Tokenizers are fun."

ids = tokenizer.encode(text)                       # text -> list of int IDs
pieces = tokenizer.convert_ids_to_tokens(ids)      # IDs  -> subword strings
print("\n[1] tokenization")
print("  IDs:     ", ids)
print("  Pieces:  ", pieces)                        # note the Ġ / 'Ċ' space markers
print("  Decoded: ", tokenizer.decode(ids))         # round trip back to text

# Watch how words split into subwords (BPE):
for word in ["cat", "tokenization", "antidisestablishmentarianism", "🤗", "⅍"]:
    print(f"  {word!r:35} -> {tokenizer.tokenize(word)}")

# The *callable* form returns model-ready tensors + an attention mask:
enc = tokenizer(text, return_tensors="pt")
print("  callable form keys:", list(enc.keys()))    # input_ids, attention_mask
print("  shape:", enc.input_ids.shape)
print(" content:")
print(enc.input_ids)
print(enc.attention_mask)


[1] tokenization
  IDs:      [9707, 11, 1879, 0, 9660, 12230, 525, 2464, 13]
  Pieces:   ['Hello', ',', 'Ġworld', '!', 'ĠToken', 'izers', 'Ġare', 'Ġfun', '.']
  Decoded:  Hello, world! Tokenizers are fun.
  'cat'                               -> ['cat']
  'tokenization'                      -> ['token', 'ization']
  'antidisestablishmentarianism'      -> ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']
  '🤗'                                 -> ['ðŁ¤Ĺ']
  '⅍'                                 -> ['â', 'ħį']
  callable form keys: ['input_ids', 'attention_mask']
  shape: torch.Size([1, 9])
 content:
tensor([[ 9707,    11,  1879,     0,  9660, 12230,   525,  2464,    13]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [3]:
enc.input_ids.dtype

torch.int64

In [10]:
# ======================================================================
# 2. CHAT TEMPLATE  (essential for any *-Instruct model)
# ======================================================================
# Instruct models were trained on a specific role-tagged format. Don't
# hand-build it — apply_chat_template inserts the right <|im_start|> markers.
messages = [
    {"role": "system", "content": "You are a terse, helpful assistant."},
    {"role": "user", "content": "Give me one fun fact about octopuses."},
]

formatted = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print("\n[2] chat template (raw string the model actually sees):")
print(formatted)
# add_generation_prompt=True appends the opening of the assistant turn,
# cueing the model to start its reply rather than continue the user's.


[2] chat template (raw string the model actually sees):
<|im_start|>system
You are a terse, helpful assistant.<|im_end|>
<|im_start|>user
Give me one fun fact about octopuses.<|im_end|>
<|im_start|>assistant



In [11]:
# ======================================================================
# 3. GENERATION / DECODING STRATEGIES
# ======================================================================
# transformers 5.x returns a BatchEncoding (dict-like) here, not a bare tensor,
# so grab the input_ids tensor explicitly. Shape: (batch=1, seq_len).
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).input_ids.to(DEVICE)

In [15]:
import logging
logging.basicConfig(level=logging.INFO)

def generate(**kwargs):
    """Generate and return ONLY the newly produced text (prompt sliced off)."""
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=80,
            pad_token_id=tokenizer.eos_token_id,  # silences a pad-token warning
            **kwargs,
        )
        #logging.info(out)
    new_tokens = out[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("\n[3] decoding strategies")
# Greedy: deterministic, always takes the highest-probability token.
print("\n  [greedy]\n ", generate(do_sample=False))

logging.getLogger().setLevel(logging.DEBUG)

# Temperature sampling: higher temp = flatter distribution = more random/creative.
print("\n  [sampling T=0.8, top_p=0.9]\n ",
      generate(do_sample=True, temperature=0.8, top_p=0.9))

# Low temperature ≈ near-greedy, more focused.
print("\n  [sampling T=0.2]\n ", generate(do_sample=True, temperature=0.2))

# Beam search: keeps several hypotheses, picks the highest-scoring sequence.
print("\n  [beam search, 4 beams]\n ", generate(num_beams=4, do_sample=False))


[3] decoding strategies

  [greedy]
  Octopuses have the ability to change color and shape based on their environment, allowing them to blend in with their surroundings and avoid predators. They also have highly developed sensory organs that allow them to detect changes in their environment, such as touch or smell. Additionally, octopuses can communicate using a complex system of tentacles called "soma," which they use to convey information between individuals within their

  [sampling T=0.8, top_p=0.9]
  Here's a fun fact: Octopuses can sing! They make a loud, shrill "hah" sound that they use to communicate with each other and attract mates.

  [sampling T=0.2]
  Octopuses have the ability to change color and shape based on their environment, allowing them to blend in with their surroundings and avoid predators. They also have excellent eyesight and hearing, which they use for hunting and communication.

  [beam search, 4 beams]
  One fun fact about octopuses is that they are the onl

In [16]:
for _ in range(5):
    print("\n  [sampling T=0.8, top_p=0.9]\n ",
        generate(do_sample=True, temperature=0.8, top_p=0.9))


  [sampling T=0.8, top_p=0.9]
  Octopuses have the ability to move using their arms and legs, which they can rotate around and make precise movements with. They also have a highly developed sense of smell that helps them locate food in dense waters. Additionally, octopuses are known for their bright colors and iridescent scales, which help them blend into their surroundings and avoid predators.

  [sampling T=0.8, top_p=0.9]
  Sure! Octopuses have the ability to manipulate their body shape and size to match their surroundings, something that some animals, including humans, have difficulty doing. This is called "reptilian" or "mammalian adaptation." It allows them to quickly change position in order to hide from predators or find food more easily.

  [sampling T=0.8, top_p=0.9]
  Octopuses are the only marine invertebrates that can climb and crawl on land. They have been known to take up residence in abandoned aquariums or as decorations for fish tanks.

  [sampling T=0.8, top_p=0.9]
 

In [17]:
for _ in range(5):
    print("\n  [sampling T=0.2]\n ", generate(do_sample=True, temperature=0.2))


  [sampling T=0.2]
  Octopuses have the ability to change color and shape based on their environment, allowing them to blend in with their surroundings and avoid predators.

  [sampling T=0.2]
  Octopuses have the ability to move their arms in a way that allows them to "swim" underwater!

  [sampling T=0.2]
  Octopuses have the ability to change color and shape based on their environment or emotions. They can also communicate using chemical signals that they release from their skin.

  [sampling T=0.2]
  Octopuses have the ability to change color and shape based on their environment, allowing them to blend in with their surroundings and avoid predators. They also have highly developed sensory organs that allow them to detect prey and threats from great distances.

  [sampling T=0.2]
  Octopuses have the ability to move their arms in different directions and can even change direction by rotating them around their body. They also have a unique sense of smell that allows them to detect p

In [19]:
for _ in range(5):
    print("\n  [beam search, 4 beams]\n ", generate(num_beams=4, do_sample=True))


  [beam search, 4 beams]
  One fun fact about octopuses is that they are the only animals that can sing!

  [beam search, 4 beams]
  One fun fact about octopuses is that they are the only animals that can sing!

  [beam search, 4 beams]
  One fun fact about octopuses is that they are the only animals that can sing!

  [beam search, 4 beams]
  One fun fact about octopuses is that they have the ability to change their skin color to blend in with their surroundings and avoid predators.

  [beam search, 4 beams]
  One fun fact about octopuses is that they are the only animals that can sing!


In [23]:
# ======================================================================
# 4. PEEK AT THE NEXT-TOKEN PROBABILITY DISTRIBUTION
# ======================================================================
# A forward pass (no generation) gives raw logits; softmax -> probabilities.
with torch.no_grad():
    logits = model(inputs).logits        # shape: (batch, seq_len, vocab)
print(logits.shape)
print(torch.softmax(logits, dim=-1).sum(dim=-1))
next_token_logits = logits[0, -1]        # distribution over the token AFTER the prompt
probs = torch.softmax(next_token_logits, dim=-1)

topk = torch.topk(probs, 10)
print("\n[4] top-10 candidate next tokens:")
for prob, tid in zip(topk.values, topk.indices):
    print(f"  {prob.item():6.2%} {tid} {tokenizer.decode([int(tid)])!r}")

torch.Size([1, 31, 151936])
tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]],
       dtype=torch.bfloat16)

[4] top-10 candidate next tokens:
  43.16% 18053 'Oct'
  23.14% 3966 'One'
   5.83% 2082 'An'
   5.83% 39814 'Sure'
   4.00% 8420 'Here'
   4.00% 785 'The'
   2.76% 46 'O'
   1.01% 6986 'Did'
   1.01% 32 'A'
   0.90% 9454 'Yes'


In [21]:
# ======================================================================
# 5. STREAMING OUTPUT (token by token, as it's produced)
# ======================================================================
from transformers import TextStreamer

print("\n[5] streamed generation:")
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
with torch.no_grad():
    _ = model.generate(
        inputs, max_new_tokens=80, do_sample=True, temperature=0.7,
        pad_token_id=tokenizer.eos_token_id, streamer=streamer,
    )


[5] streamed generation:
Octopuses have the ability to breathe underwater using their mantle cavity and use their tentacles for feeding and communication. They can also change color and pattern in response to their environment.


In [24]:
# ======================================================================
# 6. BONUS: how an SFT (prompt, target) example gets tokenized
# ======================================================================
# In SFT you tokenize prompt + target together but MASK the prompt tokens in
# the loss (label = -100), so cross-entropy only lands on the target tokens.
prompt_msgs = [{"role": "user", "content": "Capital of France?"}]
prompt_text = tokenizer.apply_chat_template(
    prompt_msgs, tokenize=False, add_generation_prompt=True
)
print("promt text:")
print(prompt_text)
target_text = "Paris."

prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
print(prompt_ids)
print(len(prompt_ids))
target_ids = tokenizer(target_text, add_special_tokens=False).input_ids
print(target_ids)
print(len(target_ids))

input_ids = prompt_ids + target_ids
labels = [-100] * len(prompt_ids) + target_ids   # -100 == ignored by the loss

print("\n[6] SFT-style example")
print(f"  prompt tokens: {len(prompt_ids)}, target tokens: {len(target_ids)}")
print("  labels (note -100 mask over the prompt):")
print("   ", labels)

promt text:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Capital of France?<|im_end|>
<|im_start|>assistant

[151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 63593, 315, 9625, 30, 151645, 198, 151644, 77091, 198]
33
[59604, 13]
2

[6] SFT-style example
  prompt tokens: 33, target tokens: 2
  labels (note -100 mask over the prompt):
    [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 59604, 13]
